In [2]:
# Systeme de recommandation — Similarite cosinus

import math

# ================================
# 1. DONNEES : notes des utilisateurs
# ================================

# Chaque utilisateur a noté certains films (note de 1 a 5)
# 0 = film non vu

notes = {
    "Alice":   [5, 4, 0, 0, 1, 0, 3],
    "Bob":     [4, 0, 4, 0, 1, 0, 3],
    "Charlie": [0, 4, 5, 1, 0, 0, 4],
    "Diana":   [0, 0, 4, 5, 0, 3, 0],
    "Eve":     [5, 4, 0, 0, 2, 0, 3],
}

films = ["Inception", "Interstellar", "Matrix", "Dune", "Titanic", "Avatar", "Parasite"]

utilisateur_cible = "Bob"    # <- change ce nom pour tester

# ================================
# 2. CALCUL DE LA SIMILARITE COSINUS
# ================================

# La similarite cosinus mesure si deux utilisateurs
# ont les memes gouts (resultat entre 0 et 1)
#
#          A . B
# cos =  ---------
#         |A| x |B|
#
# A . B  = somme des produits terme a terme
# |A|    = racine de la somme des carres

def similarite_cosinus(a, b):
    produit_scalaire = sum(a[i] * b[i] for i in range(len(a)))
    norme_a = math.sqrt(sum(x ** 2 for x in a))
    norme_b = math.sqrt(sum(x ** 2 for x in b))

    if norme_a == 0 or norme_b == 0:
        return 0

    return produit_scalaire / (norme_a * norme_b)

# ================================
# 3. TROUVER LES UTILISATEURS SIMILAIRES
# ================================

notes_cible = notes[utilisateur_cible]
similarities = {}

for nom, note in notes.items():
    if nom != utilisateur_cible:
        score = similarite_cosinus(notes_cible, note)
        similarities[nom] = score

# Trier du plus similaire au moins similaire
similarities = dict(sorted(similarities.items(), key=lambda x: x[1], reverse=True))

print("=" * 45)
print(f"Utilisateur : {utilisateur_cible}")
print("=" * 45)

print("\nSimilarite avec les autres utilisateurs :")
print("-" * 35)
for nom, score in similarities.items():
    barre = "#" * int(score * 20)
    print(f"  {nom:<10} {score:.2f}  {barre}")

# ================================
# 4. RECOMMANDER DES FILMS
# ================================

# Pour chaque film non vu par l'utilisateur cible,
# on calcule un score moyen pondere par la similarite

scores_films = {}

for i, film in enumerate(films):
    if notes_cible[i] == 0:             # film non vu par l'utilisateur cible

        total_poids = 0
        total_score = 0

        for nom, similarite in similarities.items():
            note_autre = notes[nom][i]
            if note_autre > 0:          # l'autre utilisateur a vu ce film
                total_score += similarite * note_autre
                total_poids += similarite

        if total_poids > 0:
            scores_films[film] = total_score / total_poids

# Trier les recommandations par score
recommandations = dict(sorted(scores_films.items(), key=lambda x: x[1], reverse=True))

print(f"\nFilms deja vus par {utilisateur_cible} :")
print("-" * 35)
for i, film in enumerate(films):
    if notes_cible[i] > 0:
        print(f"  {film:<15} note : {notes_cible[i]}/5")

print(f"\nRecommandations pour {utilisateur_cible} :")
print("-" * 35)
for film, score in recommandations.items():
    etoiles = "*" * round(score)
    print(f"  {film:<15} score : {score:.2f}  {etoiles}")

print("=" * 45)

Utilisateur : Bob

Similarite avec les autres utilisateurs :
-----------------------------------
  Eve        0.65  #############
  Charlie    0.65  ############
  Alice      0.65  ############
  Diana      0.35  ######

Films deja vus par Bob :
-----------------------------------
  Inception       note : 4/5
  Matrix          note : 4/5
  Titanic         note : 1/5
  Parasite        note : 3/5

Recommandations pour Bob :
-----------------------------------
  Interstellar    score : 4.00  ****
  Avatar          score : 3.00  ***
  Dune            score : 2.40  **
